# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Display available record sets and their fields by `@id`

print("Available Record Sets and their Fields:")
for record_set in metadata.record_sets:
    print(f"- Record Set: {record_set['@id']}, Name: {record_set.get('name', '<no name>')}")
    # Each record set has a list of fields
    for field in record_set.get('fields', []):
        print(f"    - Field: {field['@id']}, Name: {field.get('name', '<no name>')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print("Record Set @ids for extraction:")
for rset in record_set_ids:
    print(rset)

# Prepare a dictionary to hold dataframes for each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for Record Set {record_set_id}")

# For the following cells, select the first record set for example analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main Record Set selected: {main_record_set_id}")
    print("Columns in the main record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets available in the metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Let's pick a numeric field for basic EDA.

# Obtain the list of columns in the main record set
df = dataframes.get(main_record_set_id)
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print("Numeric fields in main record set:", numeric_candidates)

# For demonstration, we select the first numeric column, if any
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Will analyze numeric field: {numeric_field}")
else:
    numeric_field = None
    print("No numeric fields found in this record set.")

# Set a filter threshold and demonstrate filtering/normalizing
if numeric_field is not None:
    threshold = df[numeric_field].quantile(0.5)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > median ({threshold}): {len(filtered_df)} records")
    
    # Normalize the numeric field (Z-score)
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    
    print(f"First 5 normalized values of {numeric_field}:")
    display(filtered_df[[numeric_field, norm_col]].head())
    
    # Try grouping by a categorical field
    # Find a field with a low number of unique values
    group_field_candidates = [c for c in df.columns if df[c].dtype=='object' and df[c].nunique() > 1 and df[c].nunique() <= 10]
    print(f"Candidate fields for grouping: {group_field_candidates}")
    if group_field_candidates:
        group_field = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable field found for grouping.")
else:
    print("Skipping EDA: No numeric field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If we found a numeric field, display its distribution
if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If grouping field available, display group comparison
    if 'group_field' in locals():
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
- We loaded the FAIR² dataset on protein analysis with mass spectrometry using the `mlcroissant` library.
- We programmatically listed all record sets and their fields using `@id` for robust referencing.
- We extracted tabular data for further exploration: filtering, normalization, and grouping/aggregation based on available fields.
- Basic data visualization provided insight into numeric field distributions and grouped summaries (if available).

You can further extend this notebook with domain-specific data cleaning, modeling, or advanced visualizations as needed.